# Notebook 03: Ethereum & EVM Analysis

## Overview

This notebook provides a deep dive into Ethereum's architecture, the Ethereum Virtual Machine (EVM), and the gas economics that power the network. We explore how Ethereum extends the concepts from Bitcoin (Notebook 02) with a Turing-complete execution environment, enabling smart contracts and decentralized applications.

## Prerequisites
- **Notebook 01**: Cryptographic Primitives (hash functions, digital signatures, key pairs)
- Basic Python programming
- Familiarity with blockchain fundamentals (blocks, transactions, consensus)

## Learning Objectives
1. Understand Ethereum's account model (EOA vs. Contract accounts) and how it differs from Bitcoin's UTXO model
2. Analyze Ethereum block and transaction structures
3. Comprehend the EVM as a stack-based execution environment
4. Decode smart contract bytecode into opcodes
5. Model EIP-1559's base fee adjustment mechanism and analyze gas economics
6. Calculate ETH burned under EIP-1559

## Estimated Time: 4-6 hours

---

In [ ]:
# ============================================================
# Setup: Import libraries
# ============================================================
import json
import struct
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Ethereum-specific libraries (optional -- all examples have fallback data)
try:
    from web3 import Web3
    HAS_WEB3 = True
    print("web3 loaded successfully.")
except ImportError:
    HAS_WEB3 = False
    print("web3 not available -- using fallback data for all examples.")

try:
    from eth_utils import (
        is_checksum_address,
        to_checksum_address,
        is_address,
        keccak,
    )
    HAS_ETH_UTILS = True
    print("eth_utils loaded successfully.")
except ImportError:
    HAS_ETH_UTILS = False
    print("eth_utils not available -- using pure-Python fallback.")

try:
    from eth_account import Account
    HAS_ETH_ACCOUNT = True
    print("eth_account loaded successfully.")
except ImportError:
    HAS_ETH_ACCOUNT = False
    print("eth_account not available -- using fallback data.")

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("\nSetup complete.")

---
# Part 1: Ethereum Basics

Ethereum extends Bitcoin's vision of decentralized money into a **decentralized computing platform**. Where Bitcoin uses a scripting language that is intentionally limited, Ethereum provides the EVM -- a Turing-complete virtual machine capable of executing arbitrary programs called **smart contracts**.

Key differences from Bitcoin:
| Feature | Bitcoin | Ethereum |
|---------|---------|----------|
| State model | UTXO | Account-based |
| Scripting | Bitcoin Script (limited) | EVM bytecode (Turing-complete) |
| Block time | ~10 minutes | ~12 seconds (post-Merge) |
| Native currency | BTC | ETH |
| Fee model | Fee market (simple) | EIP-1559 (base fee + tip) |

## 1.1 Connecting to Ethereum (with Fallback)

We attempt to connect to a public Ethereum node. If unavailable, we use **complete hardcoded fallback data** so that all subsequent cells work offline.

In [ ]:
# Attempt connection to a public Ethereum RPC endpoint
CONNECTED = False

if HAS_WEB3:
    try:
        w3 = Web3(Web3.HTTPProvider("https://ethereum-rpc.publicnode.com", request_kwargs={'timeout': 5}))
        if w3.is_connected():
            CONNECTED = True
            latest = w3.eth.block_number
            print(f"Connected to Ethereum. Latest block: {latest:,}")
        else:
            print("Connection failed -- using fallback data.")
    except Exception as e:
        print(f"Connection error: {e}\nUsing fallback data.")
else:
    print("web3 not installed -- using fallback data.")

if not CONNECTED:
    print("All examples will use hardcoded sample data (no internet required).")

## 1.2 Wei, Gwei, and Ether Conversions

Ethereum's native currency (ETH) uses denominations analogous to cents and dollars:

| Unit | Wei Value | Common Use |
|------|-----------|------------|
| Wei | 1 | Smallest unit |
| Gwei | 10^9 | Gas prices |
| Ether | 10^18 | Account balances, transfers |

All arithmetic on-chain is performed in **Wei** (integers only -- no floating point).

In [ ]:
# Pure math -- no API needed
WEI_PER_GWEI = 10**9
WEI_PER_ETHER = 10**18

def wei_to_gwei(wei):
    return wei / WEI_PER_GWEI

def wei_to_ether(wei):
    return wei / WEI_PER_ETHER

def ether_to_wei(ether):
    return int(ether * WEI_PER_ETHER)

def gwei_to_wei(gwei):
    return int(gwei * WEI_PER_GWEI)

# Examples
print("=== Wei / Gwei / Ether Conversions ===")
print(f"1 Ether          = {WEI_PER_ETHER:,} Wei")
print(f"1 Ether          = {WEI_PER_ETHER // WEI_PER_GWEI:,} Gwei")
print(f"30 Gwei          = {gwei_to_wei(30):,} Wei")
print(f"1,500,000 Gwei   = {wei_to_ether(gwei_to_wei(1_500_000)):.4f} Ether")
print(f"0.05 Ether       = {ether_to_wei(0.05):,} Wei")

# Typical gas price scenario
gas_price_gwei = 25  # 25 Gwei
gas_used = 21_000     # simple ETH transfer
fee_wei = gwei_to_wei(gas_price_gwei) * gas_used
print(f"\nTransaction fee for simple transfer:")
print(f"  Gas price: {gas_price_gwei} Gwei")
print(f"  Gas used:  {gas_used:,}")
print(f"  Fee:       {fee_wei:,} Wei = {wei_to_ether(fee_wei):.6f} ETH")

## 1.3 Account Types: EOA vs. Contract

Ethereum has **two types of accounts**:

### Externally Owned Accounts (EOA)
- Controlled by a private key
- Can initiate transactions
- Has a balance (in Wei) and a nonce (transaction count)
- No associated code

### Contract Accounts
- Controlled by their deployed code (no private key)
- Cannot initiate transactions (only respond to them)
- Has a balance, a nonce, **code**, and **storage**
- Created via a contract creation transaction

Both account types share the same 20-byte (160-bit) address format.

In [ ]:
# Sample Ethereum addresses (well-known public addresses)
SAMPLE_ADDRESSES = {
    "Vitalik Buterin (EOA)": "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    "Ethereum Foundation (EOA)": "0xde0B295669a9FD93d5F28D9Ec85E40f4cb697BAe",
    "USDT Contract": "0xdAC17F958D2ee523a2206206994597C13D831ec7",
    "Uniswap V2 Router (Contract)": "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",
    "WETH Contract": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2",
}

print("=== Sample Ethereum Addresses ===")
for label, addr in SAMPLE_ADDRESSES.items():
    acct_type = "Contract" if "Contract" in label else "EOA"
    print(f"  {label}")
    print(f"    Address: {addr}")
    print(f"    Type:    {acct_type}")
    print(f"    Length:  {len(bytes.fromhex(addr[2:]))} bytes ({len(addr[2:])} hex chars)")
    print()

## 1.4 Address Checksum Validation (EIP-55)

Ethereum uses a **mixed-case checksum encoding** defined in [EIP-55](https://eips.ethereum.org/EIPS/eip-55). The checksum is embedded in the capitalization of the hex characters:

1. Take the lowercase hex address (without `0x`)
2. Hash it with Keccak-256
3. For each character in the address: if the corresponding nibble in the hash is >= 8, capitalize it

In [ ]:
import hashlib

def keccak256(data: bytes) -> bytes:
    """Pure-Python Keccak-256 using hashlib (available in Python 3.6+)."""
    k = hashlib.new('sha3_256')  # Note: Python's sha3_256 is standard SHA-3, not Keccak
    # For true Keccak we need pysha3 or eth_utils; we'll use eth_utils if available
    if HAS_ETH_UTILS:
        return keccak(data)
    # Fallback: use a known precomputed result for demonstration
    k.update(data)
    return k.digest()

def to_checksum_address_manual(address: str) -> str:
    """Implement EIP-55 checksum encoding."""
    addr = address.lower().replace('0x', '')
    addr_hash = keccak256(addr.encode('utf-8')).hex()
    
    checksummed = '0x'
    for i, char in enumerate(addr):
        if char in '0123456789':
            checksummed += char
        elif int(addr_hash[i], 16) >= 8:
            checksummed += char.upper()
        else:
            checksummed += char.lower()
    return checksummed

# Validate using eth_utils if available, otherwise our manual implementation
test_addr = "0xd8da6bf26964af9d7eed9e03e53415d37aa96045"  # lowercase Vitalik's address

if HAS_ETH_UTILS:
    checksummed = to_checksum_address(test_addr)
    print(f"Original (lowercase): {test_addr}")
    print(f"Checksummed (EIP-55): {checksummed}")
    print(f"Is valid checksum:    {is_checksum_address(checksummed)}")
    
    # Demonstrate that a wrong checksum is detected
    bad_addr = checksummed[:5] + checksummed[5].swapcase() + checksummed[6:]
    print(f"\nCorrupted address:    {bad_addr}")
    print(f"Is valid checksum:    {is_checksum_address(bad_addr)}")
else:
    # Fallback demo with known result
    known_checksum = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"
    print(f"Original (lowercase): {test_addr}")
    print(f"Known checksum:       {known_checksum}")
    print(f"\nEIP-55 Algorithm:")
    print(f"  1. Hash lowercase address with Keccak-256")
    print(f"  2. For each hex char: capitalize if hash nibble >= 8")
    print(f"  3. Prepend '0x'")
    print(f"\nThis catches ~99.98% of single-character errors in addresses.")

---
# Part 2: Block and Transaction Structure

Ethereum blocks contain significantly more metadata than Bitcoin blocks due to the account-based state model and EVM execution.

## 2.1 Ethereum Block Structure

An Ethereum block header includes:
- **parentHash**: Hash of the parent block
- **stateRoot**: Root of the world state trie (all account balances, storage, code)
- **transactionsRoot**: Root of the transactions trie
- **receiptsRoot**: Root of the receipts trie (execution results)
- **logsBloom**: Bloom filter for efficient log searching
- **gasLimit / gasUsed**: Block gas capacity and actual usage
- **baseFeePerGas**: EIP-1559 base fee (burned)
- **timestamp, number, nonce, difficulty** (pre-Merge), etc.

In [ ]:
# Hardcoded sample block data (Ethereum mainnet block 17,000,000 -- April 2023)
SAMPLE_BLOCK = {
    "number": 17_000_000,
    "hash": "0x7a8b2e6f4c1d3a5b9e0f2c4d6a8b0e2f4a6c8d0e2f4a6b8c0d2e4f6a8b0c2d",
    "parentHash": "0x3c1e4f6a8b0d2e4f6a8b0c2d4e6f8a0b2c4d6e8f0a2b4c6d8e0f2a4b6c8d0e",
    "timestamp": 1681338455,
    "miner": "0x388C818CA8B9251b393131C08a736A67ccB19297",  # Lido
    "gasLimit": 30_000_000,
    "gasUsed": 12_458_732,
    "baseFeePerGas": 28_735_172_859,  # ~28.7 Gwei
    "transactionCount": 143,
    "size": 89_247,  # bytes
    "stateRoot": "0xa1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7f8a9b0c1d2e3f4a5b6c7d8e9f0a1b2",
    "receiptsRoot": "0xf0e1d2c3b4a5f6e7d8c9b0a1f2e3d4c5b6a7f8e9d0c1b2a3f4e5d6c7b8a9f0e1",
    "difficulty": 0,  # Post-Merge: always 0
}

print("=== Ethereum Block #{:,} ===".format(SAMPLE_BLOCK['number']))
print(f"  Timestamp:        {SAMPLE_BLOCK['timestamp']} (Unix)")
print(f"  Validator/Miner:  {SAMPLE_BLOCK['miner']}")
print(f"  Gas Limit:        {SAMPLE_BLOCK['gasLimit']:,}")
print(f"  Gas Used:         {SAMPLE_BLOCK['gasUsed']:,} ({SAMPLE_BLOCK['gasUsed']/SAMPLE_BLOCK['gasLimit']*100:.1f}%)")
print(f"  Base Fee:         {wei_to_gwei(SAMPLE_BLOCK['baseFeePerGas']):.2f} Gwei")
print(f"  Transactions:     {SAMPLE_BLOCK['transactionCount']}")
print(f"  Size:             {SAMPLE_BLOCK['size']:,} bytes")
print(f"  Difficulty:       {SAMPLE_BLOCK['difficulty']} (post-Merge)")
print(f"  Block Hash:       {SAMPLE_BLOCK['hash'][:20]}...")
print(f"  Parent Hash:      {SAMPLE_BLOCK['parentHash'][:20]}...")
print(f"  State Root:       {SAMPLE_BLOCK['stateRoot'][:20]}...")

## 2.2 Bitcoin vs. Ethereum Block Comparison

Reference: **Notebook 02** (Bitcoin Blockchain Analysis)

| Field | Bitcoin | Ethereum |
|-------|---------|----------|
| State model | UTXO set (implicit) | State trie (explicit `stateRoot`) |
| Block size limit | ~1-4 MB (weight) | Gas limit (~30M gas) |
| Merkle tree | Transactions only | Transactions, Receipts, State |
| Execution results | Not stored | `receiptsRoot` + logs |
| Fee mechanism | Simple fee market | EIP-1559 (base + priority fee) |
| Difficulty | Proof-of-Work | 0 (post-Merge Proof-of-Stake) |
| Block time | ~10 minutes | ~12 seconds |

In [ ]:
# Visual comparison of block throughput
comparison = pd.DataFrame({
    'Metric': ['Block Time (s)', 'Blocks/Day', 'Avg Txns/Block', 'Avg Txns/Day',
               'Max Block Size', 'Header Fields'],
    'Bitcoin': ['600', '144', '~2,500', '~360,000',
                '~4 MB (weight)', '6'],
    'Ethereum': ['12', '7,200', '~150', '~1,080,000',
                 '~30M gas', '15+'],
})
print("=== Block Structure Comparison ===")
print(comparison.to_string(index=False))

## 2.3 Transaction Structure and Receipts

Ethereum transactions include:
- **from**: Sender (EOA)
- **to**: Recipient (EOA or contract; `null` for contract creation)
- **value**: ETH to transfer (in Wei)
- **input/data**: Calldata (function selector + encoded arguments)
- **gas**: Gas limit for this transaction
- **maxFeePerGas / maxPriorityFeePerGas**: EIP-1559 fee parameters
- **nonce**: Sender's transaction count

In [ ]:
# Hardcoded sample transaction (a Uniswap V2 swap)
SAMPLE_TX = {
    "hash": "0xa1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7f8a9b0c1d2e3f4a5b6c7d8e9f0a1b2",
    "blockNumber": 17_000_000,
    "from": "0x28C6c06298d514Db089934071355E5743bf21d60",  # Binance Hot Wallet
    "to": "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",    # Uniswap V2 Router
    "value": 1_000_000_000_000_000_000,  # 1 ETH
    "gas": 250_000,
    "maxFeePerGas": 35_000_000_000,       # 35 Gwei
    "maxPriorityFeePerGas": 2_000_000_000, # 2 Gwei
    "nonce": 14_582,
    "type": 2,  # EIP-1559
    "input": "0x7ff36ab5"
             "0000000000000000000000000000000000000000000000000de0b6b3a7640000"
             "0000000000000000000000000000000000000000000000000000000000000080"
             "00000000000000000000000028c6c06298d514db089934071355e5743bf21d60"
             "0000000000000000000000000000000000000000000000000000000064420b37",
}

# Hardcoded sample receipt
SAMPLE_RECEIPT = {
    "transactionHash": SAMPLE_TX["hash"],
    "status": 1,  # 1 = success, 0 = revert
    "gasUsed": 152_847,
    "effectiveGasPrice": 30_735_172_859,  # baseFee + priority fee actually paid
    "logs": [
        {"address": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2",
         "topics": ["0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"],
         "data": "0x0000000000000000000000000000000000000000000000000de0b6b3a7640000"},
        {"address": "0xdAC17F958D2ee523a2206206994597C13D831ec7",
         "topics": ["0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"],
         "data": "0x000000000000000000000000000000000000000000000000000000003b9aca00"},
    ],
    "cumulativeGasUsed": 5_234_891,
}

print("=== Sample Transaction ===")
print(f"  Hash:              {SAMPLE_TX['hash'][:20]}...")
print(f"  Block:             {SAMPLE_TX['blockNumber']:,}")
print(f"  From:              {SAMPLE_TX['from']}")
print(f"  To:                {SAMPLE_TX['to']}")
print(f"  Value:             {wei_to_ether(SAMPLE_TX['value'])} ETH")
print(f"  Gas Limit:         {SAMPLE_TX['gas']:,}")
print(f"  Max Fee:           {wei_to_gwei(SAMPLE_TX['maxFeePerGas']):.1f} Gwei")
print(f"  Priority Fee:      {wei_to_gwei(SAMPLE_TX['maxPriorityFeePerGas']):.1f} Gwei")
print(f"  Nonce:             {SAMPLE_TX['nonce']:,}")
print(f"  Type:              {SAMPLE_TX['type']} (EIP-1559)")
print(f"  Input data length: {len(SAMPLE_TX['input'])} hex chars")

print("\n=== Transaction Receipt ===")
print(f"  Status:            {'Success' if SAMPLE_RECEIPT['status'] == 1 else 'Reverted'}")
print(f"  Gas Used:          {SAMPLE_RECEIPT['gasUsed']:,} ({SAMPLE_RECEIPT['gasUsed']/SAMPLE_TX['gas']*100:.1f}% of limit)")
print(f"  Effective Price:   {wei_to_gwei(SAMPLE_RECEIPT['effectiveGasPrice']):.2f} Gwei")
fee_eth = wei_to_ether(SAMPLE_RECEIPT['gasUsed'] * SAMPLE_RECEIPT['effectiveGasPrice'])
print(f"  Total Fee:         {fee_eth:.6f} ETH")
print(f"  Event Logs:        {len(SAMPLE_RECEIPT['logs'])}")

## 2.4 Decoding Transaction Input Data

The `input` field of a contract call contains:
- **Bytes 0-3** (first 4 bytes / 8 hex chars after `0x`): **Function selector** -- the first 4 bytes of the Keccak-256 hash of the function signature
- **Bytes 4+**: ABI-encoded arguments (32-byte words)

For example, `0x7ff36ab5` is the selector for `swapExactETHForTokens(uint256,address[],address,uint256)`.

In [ ]:
# Extract function selector and decode arguments
input_data = SAMPLE_TX["input"]

# Function selector = first 4 bytes (8 hex chars after '0x')
selector = input_data[:10]  # '0x' + 8 hex chars
args_hex = input_data[10:]  # remaining data

# Split arguments into 32-byte (64-char) words
arg_words = [args_hex[i:i+64] for i in range(0, len(args_hex), 64)]

# Known function selectors (common DeFi functions)
KNOWN_SELECTORS = {
    "0xa9059cbb": "transfer(address,uint256)",
    "0x095ea7b3": "approve(address,uint256)",
    "0x23b872dd": "transferFrom(address,address,uint256)",
    "0x7ff36ab5": "swapExactETHForTokens(uint256,address[],address,uint256)",
    "0x38ed1739": "swapExactTokensForTokens(uint256,uint256,address[],address,uint256)",
    "0x18cbafe5": "swapExactTokensForETH(uint256,uint256,address[],address,uint256)",
    "0x3593564c": "execute(bytes,bytes[],uint256)",  # Uniswap Universal Router
    "0x70a08231": "balanceOf(address)",
    "0xdd62ed3e": "allowance(address,address)",
}

print("=== Decoding Transaction Input Data ===")
print(f"  Function Selector: {selector}")
print(f"  Decoded Function:  {KNOWN_SELECTORS.get(selector, 'Unknown')}")
print(f"  Number of 32-byte argument words: {len(arg_words)}")
print("\n  Argument words (raw hex):")
for i, word in enumerate(arg_words):
    # Try to interpret as uint256
    val = int(word, 16)
    print(f"    [{i}] 0x{word}")
    if val < 2**160 and val > 0:
        print(f"         -> Could be address: 0x{word[-40:]}")
    elif val > 0:
        print(f"         -> As uint256: {val:,}")

## 2.5 Gas Used vs. Gas Limit

- **Gas Limit**: The maximum amount of gas the sender is willing to pay. Set by the sender.
- **Gas Used**: The actual gas consumed during execution. Unused gas is refunded.
- If execution requires more gas than the limit, the transaction **reverts** and the gas is still consumed.

Common gas costs:
| Operation | Gas Cost |
|-----------|----------|
| Simple ETH transfer | 21,000 |
| ERC-20 transfer | ~45,000-65,000 |
| Uniswap swap | ~120,000-180,000 |
| Contract deployment | 500,000-5,000,000+ |
| SSTORE (new slot) | 20,000 |
| SSTORE (existing) | 5,000 |

In [ ]:
# Analyze gas usage across sample transactions
SAMPLE_TXS_GAS = pd.DataFrame({
    'tx_type': ['ETH Transfer', 'ETH Transfer', 'ERC-20 Transfer', 'ERC-20 Transfer',
                'ERC-20 Approve', 'Uniswap Swap', 'Uniswap Swap', 'Uniswap Swap',
                'NFT Mint', 'NFT Mint', 'Contract Deploy', 'Multi-hop Swap',
                'Aave Deposit', 'Aave Borrow'],
    'gas_limit': [21000, 21000, 65000, 80000,
                  55000, 250000, 300000, 200000,
                  350000, 400000, 3000000, 500000,
                  350000, 450000],
    'gas_used': [21000, 21000, 51438, 65213,
                 46109, 152847, 187234, 143562,
                 198432, 245891, 1847293, 312456,
                 231094, 298765],
})

SAMPLE_TXS_GAS['efficiency'] = (SAMPLE_TXS_GAS['gas_used'] / SAMPLE_TXS_GAS['gas_limit'] * 100).round(1)
SAMPLE_TXS_GAS['wasted_gas'] = SAMPLE_TXS_GAS['gas_limit'] - SAMPLE_TXS_GAS['gas_used']

print("=== Gas Usage Analysis ===")
print(SAMPLE_TXS_GAS.to_string(index=False))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gas used by type
avg_gas = SAMPLE_TXS_GAS.groupby('tx_type')['gas_used'].mean().sort_values()
ax1.barh(avg_gas.index, avg_gas.values, color='steelblue')
ax1.set_xlabel('Average Gas Used')
ax1.set_title('Average Gas Used by Transaction Type')
ax1.ticklabel_format(axis='x', style='scientific', scilimits=(0,0))

# Efficiency distribution
ax2.hist(SAMPLE_TXS_GAS['efficiency'], bins=10, color='coral', edgecolor='black')
ax2.set_xlabel('Gas Efficiency (%)')
ax2.set_ylabel('Count')
ax2.set_title('Gas Limit Efficiency (Used / Limit)')
ax2.axvline(x=100, color='green', linestyle='--', label='Perfect efficiency')
ax2.legend()

plt.tight_layout()
plt.show()

---
# Part 3: EVM Fundamentals

## 3.1 The EVM as a Stack-Based Machine

The **Ethereum Virtual Machine (EVM)** is a quasi-Turing-complete, stack-based virtual machine that executes smart contract bytecode.

### Key Properties
- **Stack-based**: Operations push/pop values from a stack (max depth: 1024)
- **Word size**: 256 bits (32 bytes) -- matches Ethereum's native integer size
- **Memory**: Byte-addressable, volatile (cleared after each call)
- **Storage**: 256-bit key-value store, persistent (written to blockchain)
- **Deterministic**: Same input always produces same output on every node
- **Gas-metered**: Every operation costs gas to prevent infinite loops

### Execution Model
```
Transaction -> EVM -> Read bytecode -> Execute opcodes -> Modify state
                         |                 |
                    Program Counter     Stack, Memory, Storage
```

The EVM processes one opcode at a time, advancing the program counter. Each opcode has a fixed gas cost (some are dynamic based on operands).

## 3.2 Common EVM Opcodes

Each opcode is a single byte (0x00 - 0xFF). Below is a reference of the most common opcodes with their gas costs.

In [ ]:
# Complete EVM opcode dictionary
# Format: hex_byte -> (mnemonic, gas_cost, description, stack_input, stack_output)
EVM_OPCODES = {
    # Stop and Arithmetic
    0x00: ("STOP",       0,   "Halt execution",                    0, 0),
    0x01: ("ADD",        3,   "Addition",                          2, 1),
    0x02: ("MUL",        5,   "Multiplication",                    2, 1),
    0x03: ("SUB",        3,   "Subtraction",                       2, 1),
    0x04: ("DIV",        5,   "Integer division",                  2, 1),
    0x05: ("SDIV",       5,   "Signed integer division",           2, 1),
    0x06: ("MOD",        5,   "Modulo",                            2, 1),
    0x07: ("SMOD",       5,   "Signed modulo",                     2, 1),
    0x08: ("ADDMOD",     8,   "(a + b) % N",                      3, 1),
    0x09: ("MULMOD",     8,   "(a * b) % N",                      3, 1),
    0x0a: ("EXP",       10,   "Exponentiation (dynamic cost)",    2, 1),
    0x0b: ("SIGNEXTEND", 5,   "Sign extend",                      2, 1),
    
    # Comparison & Bitwise Logic
    0x10: ("LT",         3,   "Less than",                         2, 1),
    0x11: ("GT",         3,   "Greater than",                      2, 1),
    0x12: ("SLT",        3,   "Signed less than",                  2, 1),
    0x13: ("SGT",        3,   "Signed greater than",               2, 1),
    0x14: ("EQ",         3,   "Equality",                          2, 1),
    0x15: ("ISZERO",     3,   "Is zero",                           1, 1),
    0x16: ("AND",        3,   "Bitwise AND",                       2, 1),
    0x17: ("OR",         3,   "Bitwise OR",                        2, 1),
    0x18: ("XOR",        3,   "Bitwise XOR",                       2, 1),
    0x19: ("NOT",        3,   "Bitwise NOT",                       1, 1),
    0x1a: ("BYTE",       3,   "Get byte from word",                2, 1),
    0x1b: ("SHL",        3,   "Shift left",                        2, 1),
    0x1c: ("SHR",        3,   "Shift right",                       2, 1),
    0x1d: ("SAR",        3,   "Arithmetic shift right",            2, 1),
    
    # SHA3 / Keccak
    0x20: ("SHA3",      30,   "Keccak-256 hash (+ dynamic)",      2, 1),
    
    # Environmental
    0x30: ("ADDRESS",    2,   "Current contract address",          0, 1),
    0x31: ("BALANCE",  100,   "Account balance (cold: 2600)",      1, 1),
    0x32: ("ORIGIN",     2,   "Transaction origin (tx.origin)",    0, 1),
    0x33: ("CALLER",     2,   "Direct caller (msg.sender)",        0, 1),
    0x34: ("CALLVALUE",  2,   "ETH sent (msg.value)",              0, 1),
    0x35: ("CALLDATALOAD", 3, "Load calldata word",                1, 1),
    0x36: ("CALLDATASIZE", 2, "Calldata size",                     0, 1),
    0x37: ("CALLDATACOPY", 3, "Copy calldata to memory",           3, 0),
    0x38: ("CODESIZE",   2,   "Code size of current contract",     0, 1),
    0x39: ("CODECOPY",   3,   "Copy code to memory",               3, 0),
    0x3a: ("GASPRICE",   2,   "Gas price of tx",                   0, 1),
    0x3b: ("EXTCODESIZE", 100, "External code size (cold: 2600)",  1, 1),
    
    # Block Information
    0x40: ("BLOCKHASH",  20,  "Hash of block",                     1, 1),
    0x41: ("COINBASE",    2,  "Block beneficiary",                 0, 1),
    0x42: ("TIMESTAMP",   2,  "Block timestamp",                   0, 1),
    0x43: ("NUMBER",      2,  "Block number",                      0, 1),
    0x44: ("DIFFICULTY",  2,  "Block difficulty / prevrandao",     0, 1),
    0x45: ("GASLIMIT",    2,  "Block gas limit",                   0, 1),
    0x46: ("CHAINID",     2,  "Chain ID",                          0, 1),
    0x48: ("BASEFEE",     2,  "Base fee (EIP-1559)",               0, 1),
    
    # Stack, Memory, Storage, Flow
    0x50: ("POP",         2,  "Remove top stack item",             1, 0),
    0x51: ("MLOAD",       3,  "Load word from memory",             1, 1),
    0x52: ("MSTORE",      3,  "Store word to memory",              2, 0),
    0x53: ("MSTORE8",     3,  "Store byte to memory",              2, 0),
    0x54: ("SLOAD",     100,  "Load from storage (cold: 2100)",    1, 1),
    0x55: ("SSTORE",   100,  "Store to storage (up to 20000)",    2, 0),
    0x56: ("JUMP",        8,  "Jump to destination",               1, 0),
    0x57: ("JUMPI",      10,  "Conditional jump",                  2, 0),
    0x58: ("PC",           2, "Program counter",                   0, 1),
    0x59: ("MSIZE",       2,  "Memory size",                       0, 1),
    0x5a: ("GAS",         2,  "Remaining gas",                     0, 1),
    0x5b: ("JUMPDEST",    1,  "Valid jump destination marker",     0, 0),
    
    # Push operations (PUSH1 through PUSH32)
    0x60: ("PUSH1",       3,  "Push 1-byte value",                 0, 1),
    0x61: ("PUSH2",       3,  "Push 2-byte value",                 0, 1),
    0x62: ("PUSH3",       3,  "Push 3-byte value",                 0, 1),
    0x63: ("PUSH4",       3,  "Push 4-byte value",                 0, 1),
    0x7f: ("PUSH32",      3,  "Push 32-byte value",                0, 1),
    
    # Dup operations
    0x80: ("DUP1",        3,  "Duplicate 1st stack item",          1, 2),
    0x81: ("DUP2",        3,  "Duplicate 2nd stack item",          2, 3),
    0x82: ("DUP3",        3,  "Duplicate 3rd stack item",          3, 4),
    
    # Swap operations
    0x90: ("SWAP1",       3,  "Swap 1st and 2nd stack items",      2, 2),
    0x91: ("SWAP2",       3,  "Swap 1st and 3rd stack items",      3, 3),
    
    # Log operations
    0xa0: ("LOG0",      375,  "Log with 0 topics",                 2, 0),
    0xa1: ("LOG1",      750,  "Log with 1 topic",                  3, 0),
    0xa2: ("LOG2",     1125,  "Log with 2 topics",                 4, 0),
    0xa3: ("LOG3",     1500,  "Log with 3 topics",                 5, 0),
    0xa4: ("LOG4",     1875,  "Log with 4 topics",                 6, 0),
    
    # System operations
    0xf0: ("CREATE",  32000,  "Create new contract",               3, 1),
    0xf1: ("CALL",      100,  "Call another contract (dynamic)",   7, 1),
    0xf2: ("CALLCODE",  100,  "Callcode (deprecated)",             7, 1),
    0xf3: ("RETURN",      0,  "Return data from call",             2, 0),
    0xf4: ("DELEGATECALL", 100, "Delegate call",                   6, 1),
    0xf5: ("CREATE2",  32000, "Create with deterministic addr",    4, 1),
    0xfa: ("STATICCALL", 100, "Static call (read-only)",           6, 1),
    0xfd: ("REVERT",      0,  "Revert execution",                  2, 0),
    0xfe: ("INVALID",     0,  "Invalid opcode",                    0, 0),
    0xff: ("SELFDESTRUCT", 5000, "Destroy contract (deprecated)",  1, 0),
}

# Fill in remaining PUSH opcodes (PUSH5-PUSH31)
for i in range(0x64, 0x80):
    n = i - 0x60 + 1
    EVM_OPCODES[i] = (f"PUSH{n}", 3, f"Push {n}-byte value", 0, 1)

# Fill in remaining DUP opcodes (DUP4-DUP16)
for i in range(0x83, 0x90):
    n = i - 0x80 + 1
    EVM_OPCODES[i] = (f"DUP{n}", 3, f"Duplicate {n}th stack item", n, n+1)

# Fill in remaining SWAP opcodes (SWAP3-SWAP16)
for i in range(0x92, 0xa0):
    n = i - 0x90 + 1
    EVM_OPCODES[i] = (f"SWAP{n}", 3, f"Swap 1st and {n+1}th stack items", n+1, n+1)

# Display opcode table
print("=== Common EVM Opcodes (selected) ===")
print(f"{'Byte':>6} {'Mnemonic':<15} {'Gas':>5}  {'Description':<35} {'In':>3} {'Out':>3}")
print("-" * 80)

display_opcodes = [0x00, 0x01, 0x02, 0x03, 0x04, 0x10, 0x14, 0x15, 0x16, 0x20,
                   0x33, 0x34, 0x35, 0x36, 0x51, 0x52, 0x54, 0x55, 0x56, 0x57,
                   0x5b, 0x60, 0x80, 0x90, 0xa1, 0xf1, 0xf3, 0xf4, 0xfd, 0xfe]
for op in display_opcodes:
    if op in EVM_OPCODES:
        name, gas, desc, inp, out = EVM_OPCODES[op]
        print(f"  0x{op:02x} {name:<15} {gas:>5}  {desc:<35} {inp:>3} {out:>3}")

print(f"\nTotal opcodes defined: {len(EVM_OPCODES)}")

## 3.3 Bytecode Decoder

Let's build a decoder that parses raw EVM bytecode into human-readable opcodes. The key insight is that `PUSH` instructions consume additional bytes as immediate data.

In [ ]:
def decode_bytecode(bytecode_hex: str) -> list:
    """
    Decode EVM bytecode into a list of (offset, opcode_name, immediate_data) tuples.
    
    PUSH1-PUSH32 opcodes consume 1-32 additional bytes as immediate data.
    All other opcodes are single-byte instructions.
    """
    # Remove '0x' prefix if present
    bytecode = bytecode_hex.replace('0x', '')
    code_bytes = bytes.fromhex(bytecode)
    
    instructions = []
    pc = 0  # program counter
    
    while pc < len(code_bytes):
        opcode = code_bytes[pc]
        
        # Look up the opcode
        if opcode in EVM_OPCODES:
            name = EVM_OPCODES[opcode][0]
        else:
            name = f"UNKNOWN(0x{opcode:02x})"
        
        # Check if this is a PUSH instruction (0x60 - 0x7f)
        if 0x60 <= opcode <= 0x7f:
            num_bytes = opcode - 0x60 + 1  # PUSH1 = 1 byte, PUSH32 = 32 bytes
            push_data = code_bytes[pc+1:pc+1+num_bytes]
            instructions.append((pc, name, push_data.hex()))
            pc += 1 + num_bytes
        else:
            instructions.append((pc, name, None))
            pc += 1
    
    return instructions


def print_bytecode(instructions: list, max_lines: int = 50):
    """Pretty-print decoded bytecode."""
    for i, (offset, name, data) in enumerate(instructions[:max_lines]):
        if data:
            print(f"  {offset:04x}: {name:<10} 0x{data}")
        else:
            print(f"  {offset:04x}: {name}")
    if len(instructions) > max_lines:
        print(f"  ... ({len(instructions) - max_lines} more instructions)")
    print(f"\nTotal instructions: {len(instructions)}")


# Test with a minimal example: PUSH1 0x42 PUSH1 0x00 SSTORE STOP
# This stores value 0x42 at storage slot 0
simple_bytecode = "6042600055" + "00"
print("=== Decoding Simple Bytecode ===")
print(f"Raw bytecode: 0x{simple_bytecode}")
print(f"Length: {len(simple_bytecode)//2} bytes\n")

instructions = decode_bytecode(simple_bytecode)
print_bytecode(instructions)

print("\nExplanation:")
print("  PUSH1 0x42  -> Push value 66 (0x42) onto stack")
print("  PUSH1 0x00  -> Push storage slot 0 onto stack")
print("  SSTORE      -> Store value (0x42) at slot (0x00)")
print("  STOP        -> Halt execution")

## 3.4 Decoding a Minimal Storage Contract

Below is the **runtime bytecode** for a simple Solidity contract that stores and retrieves a single `uint256`:

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract SimpleStorage {
    uint256 public storedValue;
    
    function set(uint256 _value) public {
        storedValue = _value;
    }
}
```

The compiler generates bytecode that:
1. Reads the function selector from calldata (first 4 bytes)
2. Compares it to known selectors (`set(uint256)` = `0x60fe47b1`, `storedValue()` = `0x2a1afcd9`)
3. Routes execution to the appropriate function body

In [ ]:
# Hardcoded runtime bytecode for SimpleStorage contract
# This is a realistic (simplified) version of what solc would produce
SIMPLE_STORAGE_BYTECODE = (
    "6080604052"          # PUSH1 0x80 PUSH1 0x40 MSTORE (free memory pointer)
    "600436106043"        # PUSH1 0x04 CALLDATASIZE LT PUSH1 0x43 (if calldata < 4 bytes, revert)
    "57"                  # JUMPI
    "6000357c0100000000"  # PUSH1 0x00 CALLDATALOAD PUSH29 0x01000... (get selector)
    "900490"              # SWAP1 DIV SWAP1 (extract first 4 bytes)
    "63"                  # PUSH4
    "60fe47b1"            # function selector for set(uint256)
    "1460"                # EQ PUSH1
    "2a"                  # jump dest offset for set()
    "57"                  # JUMPI
    "63"                  # PUSH4  
    "2a1afcd9"            # function selector for storedValue()
    "14"                  # EQ
    "6038"                # PUSH1 0x38 (jump dest for storedValue)
    "57"                  # JUMPI
    "5b"                  # JUMPDEST (fallback / revert)
    "600080fd"            # PUSH1 0x00 DUP1 REVERT
    "5b"                  # JUMPDEST (set function body)
    "60243560005500"      # PUSH1 0x24 CALLDATALOAD PUSH1 0x00 SSTORE STOP
    "5b"                  # JUMPDEST (storedValue getter)
    "60005460005260206000f3"  # SLOAD, MSTORE, RETURN 32 bytes
)

print("=== SimpleStorage Contract Bytecode ===")
print(f"Bytecode length: {len(SIMPLE_STORAGE_BYTECODE)//2} bytes")
print(f"\nRaw: 0x{SIMPLE_STORAGE_BYTECODE[:80]}...\n")

decoded = decode_bytecode(SIMPLE_STORAGE_BYTECODE)
print("Decoded instructions:")
print_bytecode(decoded)

# Analyze the bytecode
opcode_names = [name for _, name, _ in decoded]
print("\n=== Bytecode Analysis ===")
print(f"Total instructions: {len(decoded)}")
print(f"Unique opcodes used: {len(set(opcode_names))}")
freq = Counter(opcode_names)
print("\nOpcode frequency:")
for name, count in freq.most_common(10):
    print(f"  {name:<12} {count}")

---
# Part 4: Gas Analysis and EIP-1559

## 4.1 EIP-1559 Fee Model

Before EIP-1559 (London upgrade, August 2021), Ethereum used a simple first-price auction for gas. EIP-1559 introduced:

1. **Base Fee**: Algorithmically determined per block. **Burned** (removed from supply).
2. **Priority Fee (Tip)**: Optional tip to the validator. Incentivizes transaction inclusion.
3. **Max Fee**: The maximum total fee per gas the sender is willing to pay.

**Fee calculation:**
```
effective_fee = min(maxFeePerGas, baseFeePerGas + maxPriorityFeePerGas)
total_cost = gasUsed * effective_fee
burned = gasUsed * baseFeePerGas
validator_tip = gasUsed * (effective_fee - baseFeePerGas)
```

### Base Fee Adjustment
The base fee adjusts each block based on how full the previous block was relative to the **target** (50% of gas limit):

- Block > 50% full -> base fee **increases** (up to 12.5% per block)
- Block < 50% full -> base fee **decreases** (up to 12.5% per block)
- Block exactly 50% full -> base fee **unchanged**

In [ ]:
def calculate_next_base_fee(parent_base_fee: int, parent_gas_used: int, parent_gas_limit: int) -> int:
    """
    Calculate the next block's base fee per EIP-1559.
    
    Formula:
        target = parent_gas_limit // 2
        if gas_used == target: base fee unchanged
        if gas_used > target:  new_base = old_base * (1 + (gas_used - target) / target / 8)
        if gas_used < target:  new_base = old_base * (1 - (target - gas_used) / target / 8)
    
    The max change per block is 1/8 = 12.5%.
    """
    target = parent_gas_limit // 2
    
    if parent_gas_used == target:
        return parent_base_fee
    elif parent_gas_used > target:
        # Fee increases
        fee_delta = parent_base_fee * (parent_gas_used - target) // (target * 8)
        # Ensure minimum increase of 1 wei
        return parent_base_fee + max(fee_delta, 1)
    else:
        # Fee decreases
        fee_delta = parent_base_fee * (target - parent_gas_used) // (target * 8)
        return max(parent_base_fee - fee_delta, 1)  # Floor of 1 wei


# Demonstrate base fee adjustment
base_fee = 30_000_000_000  # 30 Gwei
gas_limit = 30_000_000
target = gas_limit // 2

scenarios = [
    ("Empty block (0%)", 0),
    ("25% full", gas_limit // 4),
    ("50% full (target)", target),
    ("75% full", int(gas_limit * 0.75)),
    ("100% full", gas_limit),
]

print("=== EIP-1559 Base Fee Adjustment Examples ===")
print(f"Parent base fee: {wei_to_gwei(base_fee):.2f} Gwei")
print(f"Gas limit: {gas_limit:,}  |  Target: {target:,}\n")
print(f"{'Scenario':<25} {'Gas Used':>12} {'Utilization':>12} {'New Base Fee':>14} {'Change':>10}")
print("-" * 75)
for label, gas_used in scenarios:
    new_base = calculate_next_base_fee(base_fee, gas_used, gas_limit)
    change_pct = (new_base - base_fee) / base_fee * 100
    print(f"{label:<25} {gas_used:>12,} {gas_used/gas_limit*100:>11.1f}% {wei_to_gwei(new_base):>12.2f} Gwei {change_pct:>+9.2f}%")

## 4.2 Simulating Base Fee Over 200 Blocks

Let's simulate how the base fee evolves over 200 blocks with varying network utilization patterns.

In [ ]:
np.random.seed(42)

NUM_BLOCKS = 200
GAS_LIMIT = 30_000_000
INITIAL_BASE_FEE = 20_000_000_000  # 20 Gwei

# Simulate utilization: periods of high and low demand
# Phase 1 (blocks 0-49):   moderate demand (~55-65%)
# Phase 2 (blocks 50-99):  surge in demand (~80-100%)
# Phase 3 (blocks 100-149): demand drops (~20-40%)
# Phase 4 (blocks 150-199): recovery (~45-60%)

utilization = np.concatenate([
    np.clip(np.random.normal(0.60, 0.05, 50), 0.01, 1.0),  # moderate
    np.clip(np.random.normal(0.90, 0.07, 50), 0.01, 1.0),  # surge
    np.clip(np.random.normal(0.30, 0.08, 50), 0.01, 1.0),  # drop
    np.clip(np.random.normal(0.52, 0.06, 50), 0.01, 1.0),  # recovery
])

gas_used_arr = (utilization * GAS_LIMIT).astype(int)

# Simulate base fee evolution
base_fees = [INITIAL_BASE_FEE]
eth_burned = []

for i in range(NUM_BLOCKS):
    current_base = base_fees[-1]
    gas_used = gas_used_arr[i]
    
    # Calculate ETH burned this block
    burned_wei = current_base * gas_used
    eth_burned.append(wei_to_ether(burned_wei))
    
    # Calculate next block's base fee
    next_base = calculate_next_base_fee(current_base, gas_used, GAS_LIMIT)
    base_fees.append(next_base)

base_fees_gwei = [wei_to_gwei(bf) for bf in base_fees]

# Create a comprehensive visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot 1: Base Fee
axes[0].plot(range(NUM_BLOCKS + 1), base_fees_gwei, color='blue', linewidth=1.5)
axes[0].set_ylabel('Base Fee (Gwei)')
axes[0].set_title('EIP-1559 Base Fee Simulation (200 Blocks)')
axes[0].axhline(y=base_fees_gwei[0], color='gray', linestyle='--', alpha=0.5, label='Initial base fee')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Add phase labels
for ax in axes:
    ax.axvspan(0, 50, alpha=0.05, color='green')
    ax.axvspan(50, 100, alpha=0.05, color='red')
    ax.axvspan(100, 150, alpha=0.05, color='blue')
    ax.axvspan(150, 200, alpha=0.05, color='orange')

axes[0].text(25, axes[0].get_ylim()[1]*0.9, 'Moderate', ha='center', fontsize=9)
axes[0].text(75, axes[0].get_ylim()[1]*0.9, 'Surge', ha='center', fontsize=9, color='red')
axes[0].text(125, axes[0].get_ylim()[1]*0.9, 'Drop', ha='center', fontsize=9, color='blue')
axes[0].text(175, axes[0].get_ylim()[1]*0.9, 'Recovery', ha='center', fontsize=9)

# Plot 2: Utilization
axes[1].bar(range(NUM_BLOCKS), utilization * 100, color='steelblue', alpha=0.7, width=1.0)
axes[1].axhline(y=50, color='red', linestyle='--', linewidth=1.5, label='Target (50%)')
axes[1].set_ylabel('Block Utilization (%)')
axes[1].set_title('Block Gas Utilization')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: ETH Burned
axes[2].bar(range(NUM_BLOCKS), eth_burned, color='coral', alpha=0.7, width=1.0)
axes[2].set_ylabel('ETH Burned')
axes[2].set_xlabel('Block Number')
axes[2].set_title(f'ETH Burned Per Block (Total: {sum(eth_burned):.4f} ETH)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("=== Simulation Summary ===")
print(f"Initial base fee:  {base_fees_gwei[0]:.2f} Gwei")
print(f"Final base fee:    {base_fees_gwei[-1]:.2f} Gwei")
print(f"Peak base fee:     {max(base_fees_gwei):.2f} Gwei (block {base_fees_gwei.index(max(base_fees_gwei))})")
print(f"Min base fee:      {min(base_fees_gwei):.2f} Gwei (block {base_fees_gwei.index(min(base_fees_gwei))})")
print(f"Total ETH burned:  {sum(eth_burned):.4f} ETH")
print(f"Avg ETH per block: {np.mean(eth_burned):.6f} ETH")

## 4.3 ETH Burned Per Block

Under EIP-1559, the base fee is **burned** (destroyed). This creates deflationary pressure on ETH supply, especially during high-demand periods. If the ETH burned per block exceeds the ETH issued as staking rewards (~0.05 ETH/block post-Merge), the net supply **decreases**.

In [ ]:
# Cumulative burn analysis
cumulative_burn = np.cumsum(eth_burned)

# Approximate PoS issuance (~0.05 ETH per block at ~15M ETH staked)
pos_issuance_per_block = 0.05
cumulative_issuance = np.arange(1, NUM_BLOCKS + 1) * pos_issuance_per_block
net_supply_change = cumulative_issuance - cumulative_burn

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative burn vs issuance
ax1.plot(cumulative_burn, label='ETH Burned (cumulative)', color='red', linewidth=2)
ax1.plot(cumulative_issuance, label='ETH Issued (PoS rewards)', color='green', linewidth=2)
ax1.fill_between(range(NUM_BLOCKS), cumulative_burn, cumulative_issuance,
                 where=cumulative_burn > cumulative_issuance, alpha=0.3, color='red', label='Net deflationary')
ax1.fill_between(range(NUM_BLOCKS), cumulative_burn, cumulative_issuance,
                 where=cumulative_burn <= cumulative_issuance, alpha=0.3, color='green', label='Net inflationary')
ax1.set_xlabel('Block')
ax1.set_ylabel('ETH')
ax1.set_title('Cumulative Burn vs. Issuance')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Net supply change
ax2.plot(net_supply_change, color='purple', linewidth=2)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.fill_between(range(NUM_BLOCKS), net_supply_change, 0,
                 where=np.array(net_supply_change) > 0, alpha=0.3, color='green', label='Inflationary')
ax2.fill_between(range(NUM_BLOCKS), net_supply_change, 0,
                 where=np.array(net_supply_change) <= 0, alpha=0.3, color='red', label='Deflationary')
ax2.set_xlabel('Block')
ax2.set_ylabel('Net ETH Supply Change')
ax2.set_title('Net Supply Change (Issuance - Burn)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"After {NUM_BLOCKS} blocks:")
print(f"  Total burned:   {cumulative_burn[-1]:.4f} ETH")
print(f"  Total issued:   {cumulative_issuance[-1]:.4f} ETH")
print(f"  Net change:     {net_supply_change[-1]:+.4f} ETH ({'deflationary' if net_supply_change[-1] < 0 else 'inflationary'})")

## 4.4 Gas Cost Comparison for Common Operations

Different operations on Ethereum have vastly different gas costs. Understanding these is essential for smart contract optimization.

In [ ]:
# Gas cost comparison table
gas_costs = pd.DataFrame({
    'Operation': [
        'ETH Transfer',
        'ERC-20 Transfer',
        'ERC-20 Approve',
        'Uniswap V2 Swap',
        'Uniswap V3 Swap',
        'NFT Mint (ERC-721)',
        'NFT Transfer',
        'Aave Deposit',
        'Aave Borrow',
        'Contract Deployment (small)',
        'Contract Deployment (large)',
        'ENS Registration',
        'Tornado Cash Deposit',
        'Gnosis Safe Execution',
    ],
    'Typical Gas': [
        21_000,
        65_000,
        46_000,
        150_000,
        130_000,
        200_000,
        85_000,
        250_000,
        300_000,
        500_000,
        3_000_000,
        280_000,
        1_000_000,
        150_000,
    ],
})

# Calculate costs at different gas prices
for gwei_price in [10, 30, 100]:
    gas_costs[f'Cost @ {gwei_price} Gwei (ETH)'] = (
        gas_costs['Typical Gas'] * gwei_price * 1e9 / 1e18
    ).round(6)

# Add USD cost at ~$3000/ETH for reference
eth_price_usd = 3000
gas_costs['Cost @ 30 Gwei (USD)'] = (
    gas_costs['Typical Gas'] * 30 * 1e9 / 1e18 * eth_price_usd
).round(2)

print("=== Gas Cost Comparison Table ===")
print(gas_costs.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(gas_costs['Operation'], gas_costs['Typical Gas'], color='steelblue')
ax.set_xlabel('Gas Units')
ax.set_title('Typical Gas Costs for Common Ethereum Operations')
ax.ticklabel_format(axis='x', style='scientific', scilimits=(0,0))

# Add value labels
for bar, val in zip(bars, gas_costs['Typical Gas']):
    ax.text(bar.get_width() + 20000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
# Part 5: Exercises

Complete the following exercises to reinforce your understanding of Ethereum and the EVM. Solutions can be found in `sections/03-ethereum-smart-contracts.md`.

---

## Exercise 1: Decode Function Selectors from Transaction Input Data

Given a list of transaction input data strings, extract the function selector (first 4 bytes) and look it up in the `KNOWN_SELECTORS` dictionary. Calculate what percentage of transactions have known selectors.

In [ ]:
# Exercise 1: Decode function selectors
# 
# Given these transaction inputs, extract and identify the function selectors.
# Report: (1) each selector and its function name, (2) percentage with known selectors.

EXERCISE_TX_INPUTS = [
    "0xa9059cbb000000000000000000000000dac17f958d2ee523a2206206994597c13d831ec70000000000000000000000000000000000000000000000000000000005f5e100",
    "0x095ea7b3000000000000000000000000def1c0ded9bec7f1a1670819833240f027b25eff00000000000000000000000000000000000000000000000000000000ffffffff",
    "0x7ff36ab50000000000000000000000000000000000000000000000000de0b6b3a7640000",
    "0x23b872dd000000000000000000000000abc123000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000064",
    "0x3593564c000000000000000000000000000000000000000000000000000000000000006000000000000000000000000000000000000000000000000000000000000000a0",
    "0xabcdef12000000000000000000000000000000000000000000000000000000000000002a",
    "0x70a08231000000000000000000000000d8da6bf26964af9d7eed9e03e53415d37aa96045",
    "0x18cbafe50000000000000000000000000000000000000000000000000de0b6b3a76400000000000000000000000000000000000000000000000000000000000005f5e100",
]

# YOUR CODE HERE
# Hint: selector = input_data[:10]  (includes '0x' prefix)
# Use KNOWN_SELECTORS dict from section 2.4

print("=== Exercise 1: Function Selector Decoder ===")
known_count = 0
for i, tx_input in enumerate(EXERCISE_TX_INPUTS):
    selector = tx_input[:10]
    func_name = KNOWN_SELECTORS.get(selector, "UNKNOWN")
    is_known = func_name != "UNKNOWN"
    known_count += is_known
    print(f"  TX {i+1}: {selector} -> {func_name}")

print(f"\nKnown selectors: {known_count}/{len(EXERCISE_TX_INPUTS)} ({known_count/len(EXERCISE_TX_INPUTS)*100:.0f}%)")
print("\n# TODO: Try adding more selectors to KNOWN_SELECTORS and re-run.")
print("# Challenge: Compute the selector for 'transfer(address,uint256)' using keccak256.")

## Exercise 2: Simulate EIP-1559 for Different Utilization Scenarios

Create three different utilization scenarios and compare how the base fee evolves:
1. **Steady 70%** utilization (above target)
2. **Oscillating** between 20% and 80% every 25 blocks
3. **Spike**: normal (50%) with a sudden 100% spike for 20 blocks in the middle

In [ ]:
# Exercise 2: EIP-1559 scenario comparison

def simulate_eip1559(utilization_array, initial_base_fee, gas_limit):
    """Simulate base fee evolution given a utilization array."""
    base_fees = [initial_base_fee]
    burned = []
    for util in utilization_array:
        gas_used = int(util * gas_limit)
        current_base = base_fees[-1]
        burned.append(wei_to_ether(current_base * gas_used))
        next_base = calculate_next_base_fee(current_base, gas_used, gas_limit)
        base_fees.append(next_base)
    return base_fees, burned

N = 100
INIT_BASE = 20_000_000_000  # 20 Gwei
GL = 30_000_000

# Scenario 1: Steady 70%
util_steady = np.full(N, 0.70)

# Scenario 2: Oscillating 20% <-> 80% every 25 blocks
util_oscillating = np.concatenate([np.full(25, 0.20), np.full(25, 0.80)] * (N // 50))

# Scenario 3: Normal 50% with spike to 100% at blocks 40-59
util_spike = np.full(N, 0.50)
util_spike[40:60] = 1.0

scenarios = {
    'Steady 70%': util_steady,
    'Oscillating 20/80%': util_oscillating,
    'Spike (50% + burst)': util_spike,
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
colors = ['blue', 'orange', 'red']

for (label, util), color in zip(scenarios.items(), colors):
    fees, burned = simulate_eip1559(util, INIT_BASE, GL)
    fees_gwei = [wei_to_gwei(f) for f in fees]
    
    axes[0].plot(fees_gwei, label=label, color=color, linewidth=2)
    axes[1].plot(np.cumsum(burned), label=f'{label} (total: {sum(burned):.4f} ETH)',
                 color=color, linewidth=2)

axes[0].set_ylabel('Base Fee (Gwei)')
axes[0].set_title('Base Fee Evolution: Three Scenarios')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Block')
axes[1].set_ylabel('Cumulative ETH Burned')
axes[1].set_title('Cumulative ETH Burned')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 3: Build an Opcode Frequency Analyzer

Write a function that takes bytecode, decodes it, and returns a frequency analysis of opcodes. Apply it to the sample bytecodes below and compare their profiles.

In [ ]:
# Exercise 3: Opcode frequency analyzer

def analyze_opcode_frequency(bytecode_hex: str) -> pd.DataFrame:
    """
    Decode bytecode and return a DataFrame of opcode frequencies.
    Includes: opcode name, count, percentage, and total gas cost.
    """
    instructions = decode_bytecode(bytecode_hex)
    
    # Count opcodes
    counts = Counter()
    gas_by_opcode = Counter()
    
    for offset, name, data in instructions:
        counts[name] += 1
        # Look up gas cost
        for op_byte, (op_name, gas, *_) in EVM_OPCODES.items():
            if op_name == name:
                gas_by_opcode[name] += gas
                break
    
    total = len(instructions)
    rows = []
    for name, count in counts.most_common():
        rows.append({
            'Opcode': name,
            'Count': count,
            'Percentage': f"{count/total*100:.1f}%",
            'Total Gas': gas_by_opcode.get(name, 0),
        })
    
    return pd.DataFrame(rows)


# Sample bytecodes to analyze
SAMPLE_BYTECODES = {
    "SimpleStorage": SIMPLE_STORAGE_BYTECODE,
    "Token-like": (
        "6080604052348015600f57600080fd5b50"
        "6004361060325760003560e01c8063"
        "70a0823114603757806318160ddd14605d57"
        "5b600080fd5b"
        "604360048036038101906041919060a4565b"
        "005b60536004803603810190604f919060a4565b"
        "6040518082815260200191505060405180910390f35b"
        "60636069565b60405180821515815260200191505060405180910390f35b"
        "60008054905090565b"
        "600060208284031215608357600080fd5b"
        "813590509291505056"
    ),
    "Minimal Proxy (EIP-1167)": (
        "363d3d373d3d3d363d73"
        "bebebebebebebebebebebebebebebebebebebebe"
        "5af43d82803e903d91602b57fd5bf3"
    ),
}

for name, bytecode in SAMPLE_BYTECODES.items():
    print(f"\n{'='*60}")
    print(f"=== {name} ({len(bytecode)//2} bytes) ===")
    print(f"{'='*60}")
    df = analyze_opcode_frequency(bytecode)
    print(df.to_string(index=False))

## Exercise 4: Calculate Total ETH Burned Over N Blocks

Using realistic block data (hardcoded below), calculate the total ETH burned and the average burn rate. Determine whether each block was net-inflationary or net-deflationary.

In [ ]:
# Exercise 4: ETH burn calculation from realistic block data

# Hardcoded sample of 20 consecutive blocks (realistic values)
BLOCK_DATA = pd.DataFrame({
    'block_number': range(17_000_000, 17_000_020),
    'gas_limit': [30_000_000] * 20,
    'gas_used': [
        12_458_732, 15_234_891, 29_876_543, 8_234_567, 22_345_678,
        14_567_890, 11_234_567, 27_654_321, 19_876_543, 13_456_789,
        25_678_901, 10_123_456, 18_765_432, 14_321_098, 29_012_345,
        7_654_321, 21_098_765, 16_543_210, 24_876_543, 12_345_678,
    ],
    'base_fee_gwei': [
        28.7, 27.9, 28.5, 32.1, 30.2,
        33.8, 33.1, 31.9, 35.4, 36.8,
        35.2, 38.9, 37.1, 37.5, 36.4,
        40.2, 38.5, 39.1, 38.3, 40.8,
    ],
})

# Calculate ETH burned per block
BLOCK_DATA['base_fee_wei'] = (BLOCK_DATA['base_fee_gwei'] * 1e9).astype(int)
BLOCK_DATA['eth_burned'] = BLOCK_DATA['gas_used'] * BLOCK_DATA['base_fee_wei'] / 1e18
BLOCK_DATA['utilization_pct'] = (BLOCK_DATA['gas_used'] / BLOCK_DATA['gas_limit'] * 100).round(1)

# PoS issuance per block (~0.05 ETH)
POS_ISSUANCE = 0.05
BLOCK_DATA['net_supply_change'] = POS_ISSUANCE - BLOCK_DATA['eth_burned']
BLOCK_DATA['is_deflationary'] = BLOCK_DATA['net_supply_change'] < 0

print("=== ETH Burn Analysis for 20 Blocks ===")
display_cols = ['block_number', 'utilization_pct', 'base_fee_gwei', 'eth_burned', 'net_supply_change', 'is_deflationary']
print(BLOCK_DATA[display_cols].to_string(index=False))

print(f"\n=== Summary ===")
print(f"Total ETH burned:       {BLOCK_DATA['eth_burned'].sum():.4f} ETH")
print(f"Total ETH issued:       {POS_ISSUANCE * len(BLOCK_DATA):.4f} ETH")
print(f"Net supply change:      {BLOCK_DATA['net_supply_change'].sum():+.4f} ETH")
print(f"Deflationary blocks:    {BLOCK_DATA['is_deflationary'].sum()}/{len(BLOCK_DATA)}")
print(f"Avg burn per block:     {BLOCK_DATA['eth_burned'].mean():.4f} ETH")
print(f"Avg utilization:        {BLOCK_DATA['utilization_pct'].mean():.1f}%")
print(f"\nBreakeven base fee for net-zero supply (at 50% utilization):")
breakeven_base = POS_ISSUANCE * 1e18 / (15_000_000)  # target gas = 15M
print(f"  {breakeven_base / 1e9:.2f} Gwei")

---
# Summary

## Key Takeaways

1. **Account Model**: Ethereum uses an account-based model (EOA + Contract accounts) rather than Bitcoin's UTXO model. This enables persistent state and complex logic.

2. **Block Structure**: Ethereum blocks contain state roots, receipt roots, and gas metrics that go far beyond Bitcoin's simple Merkle root of transactions.

3. **Transaction Anatomy**: Every contract interaction includes a function selector (first 4 bytes of input data) and ABI-encoded arguments.

4. **EVM**: A stack-based, gas-metered virtual machine that executes bytecode deterministically across all nodes. Each opcode has a defined gas cost.

5. **EIP-1559**: The base fee adjusts algorithmically based on block utilization. The burned base fee creates deflationary pressure, while validators receive only the priority fee (tip).

6. **Gas Economics**: Understanding gas costs is critical for both users (estimating fees) and developers (optimizing contracts).

## Further Reading

- See `sections/03-ethereum-smart-contracts.md` for detailed explanations and solutions to the exercises
- [Ethereum Yellow Paper](https://ethereum.github.io/yellowpaper/paper.pdf) -- formal EVM specification
- [EIP-1559 Specification](https://eips.ethereum.org/EIPS/eip-1559)
- [EVM Opcodes Reference](https://www.evm.codes/)
- **Next**: Notebook 04 will cover smart contract development and Solidity in depth